In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from nd2 import ND2File

from calmutils.imageio.nd2_helpers import get_z_direction

def augment_coords(coords):
    # helper function to add extra 4th column of 1s so we can just multipy with transform matrix
    return np.hstack((coords, np.ones_like(coords, shape=(len(coords), 1))))

# Correct Chromatic Aberrations for tables

This notebook will use the channel-to-channel transformations estimated with ```chromatic_aberration_estimation{_elastix}.ipynb``` and apply them to tabular data containing coordinates.

We need:
1. the saved JSON transform information from the estimation recipes
2. table(s) containing coordinates and the original channel

**Input**
1. path to a directory containing csv files
2. path to the saved transforms
3. path to write output to
4. **Parameters**: which channel to use as reference, which columns to transform, optionally channel name map if the names differ in JSON and the tables

This recipe will produce:
* tables with additional columns containing the corrected coordinates and the reference channel
* for each input table, a separate output table will be created

In [ ]:
in_path = '/Users/david/Desktop/'
transforms_path = '/Users/david/Desktop/23AM09-03/23AM09-03_003_channel_registration.json'

# subdirectories for input spots & output corrected spots
spot_subdirectory = 'spot-detection'
out_subdirectory = 'spot-detection-chromatic-shift-corrected'

# which channel the coordinates should be aligned to
reference_channel = '405-CSU-W1'

### column names of interest
# specify how to find coordinates and the channel in tables
coordinate_column_names_unit = ['z_micron', 'y_micron', 'x_micron']
# alternatively, if you do not want to use unit columns, set them to None
# coordinate_column_names_unit = None
coordinate_column_names_pixel = ['z', 'y', 'x']
channel_column_name = 'channel'

# pixel size should be zyx-array, or None (in which case it will be determined automatically)
pixel_size = None

# whether to correct z direction or not, will attempt to read z direction and image size from
# image_file column in spot files. NOTE: only works for nd2 at the moment
correct_z_direction = True

# how to name columns that are added to tables
column_name_suffix = '_shift_corrected'
reference_channel_column_name = 'shift_reference_channel'

### Channel Renaming
# if the channel names in the JSON transform file and the coordinate tables differ
# e.g. if the OC names in NIS were different or images were resaved and just have channel 0, 1, ...,
# we have to rename the channels from the JSON file to match the ones in the table
# the channel alias map should have the form: name in JSON -> name in coordinate tables

# NOTE: renaming should no longer be necessary as we clean channel names in transform estimation
# keep for now, as it may get relevant for plain TIFF files with no channel names...

# channel_aliases = {
#     '405 CSU-W1': '405-CSU-W1',
#     '488 CSU-W1': '488-CSU-W1',
#     '561 CSU-W1': '561-CSU-W1',
#     '640 CSU-W1': '640-CSU-W1'
# }

# if you do not want to rename the channels, just use an empty dict
channel_aliases = {}

In [ ]:
# make Path objects for input, output and parameter paths
in_path = Path(in_path) / spot_subdirectory
out_path = Path(in_path) / out_subdirectory
transforms_path = Path(transforms_path)

# check if enough information is given, raise Error otherwise
if coordinate_column_names_pixel is None and coordinate_column_names_unit is None:
    raise ValueError('Please specify either pixel or unit columns to transform (or both)')
if pixel_size is None and coordinate_column_names_unit is None:
    raise ValueError('You need to specify pixel size if only pixel coordinates are given')

In [ ]:
with open(transforms_path) as fd:
    transform_info = json.load(fd)

# transforms are saved as list of dicts containing channel pair and (flat) parameters
# build dict channel pair -> transform matrix
transforms = {}
for transform_info_i in transform_info['transforms']:
    
    tr = np.array(transform_info_i['parameters']).reshape(4,4)
    
    # apply channel renaming if necessary
    channels = map(lambda c: channel_aliases[c] if c in channel_aliases else c, transform_info_i['channels'])

    transforms[tuple(channels)] = tr

in_files = sorted(in_path.glob('*.csv'))
in_files

In [ ]:
# make out path if it does not exist already
if not out_path.exists():
    out_path.mkdir()

for in_file in in_files:

    df = pd.read_csv(in_file)

    # automatically determine pixel size if both pixel and unit coordinates are present
    # (we just use the first row, as we can assue it to be the same for every spot)
    pixel_size_was_none = False
    if pixel_size is None:
        pixel_size = (df[coordinate_column_names_unit].values / df[coordinate_column_names_pixel].values)[0]
        pixel_size_was_none = True

    # check if we need to flip z coordinates because reference image for parameters and current image did not have same direction
    z_flip_needed = False
    if correct_z_direction and not len(df) == 0:
        image_file = next(iter(df['image_file']))
        if Path(image_file).suffix != '.nd2':
            raise ValueError(f'z direction compensation only supported for nd2 files, got {image_file}')
        z_direction = get_z_direction(image_file)
        z_flip_needed = z_direction is not None and transform_info['z_direction'] != z_direction

        # get stack z-size in physical units
        if z_flip_needed:
            with ND2File(image_file) as fd:
                z_fov = fd.sizes['Z'] * fd.voxel_size().z

    dfs = []
    for ch, dfi in df.groupby(channel_column_name):

        # get the transform from current to reference channel
        tr = transforms[(ch, reference_channel)]

        # get coordinates to transform, 2 options:
        # 1. if we have unit columns, use those
        # 2. if we only have pixel columns, use those, multiply with pixel size
        if coordinate_column_names_unit is not None:
            coords = augment_coords(dfi[coordinate_column_names_unit].values).T
        else:
            coords = augment_coords(dfi[coordinate_column_names_pixel].values * pixel_size).T

        if z_flip_needed:
            # flip z axis, move back up by field-of-view
            coords[..., 0] *= -1.0
            coords[..., 0] += z_fov

        # transform
        coords_transformed = tr @ coords

        if z_flip_needed:
            # undo flip in transformed coords
            coords_transformed[..., 0] *= -1.0
            coords_transformed[..., 0] += z_fov

        # add transformed coordinates in world coordinate units
        if coordinate_column_names_unit is not None:
            for i, dim_name in enumerate(coordinate_column_names_unit):
                dfi[dim_name + column_name_suffix] = coords_transformed[i]

        # add transformed pixel coordinates
        if coordinate_column_names_pixel is not None:
            for i, dim_name in enumerate(coordinate_column_names_pixel):
                dfi[dim_name + column_name_suffix] = coords_transformed[i] / pixel_size[i]

        # add reference channel
        dfi[reference_channel_column_name] = reference_channel
        dfs.append(dfi)

    # combine corrected dfs
    df_corrected = pd.concat(dfs).sort_index()

    # save
    out_file = out_path / (in_file.stem + '_shift-corrected.csv')
    df_corrected.to_csv(out_file, index=False)

    # reset pixel size so it is read again for the next table
    if pixel_size_was_none:
        pixel_size = None